Install Libraries


In [4]:
!pip install groq --quiet
import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("libraries ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00
libraries ready


API_KEY="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"

In [5]:
from groq import Groq

API_KEY = "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"

client = Groq(api_key=API_KEY)

Model = "llama-3-1-8b-instant"

print(f'Groq client configured with model: {Model}')

print('Make sure API_KEY is replaced with your actual key!')




Groq client configured with model: llama-3-1-8b-instant
Make sure API_KEY is replaced with your actual key!


In [6]:
from groq import Groq

API_KEY = "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"

client = Groq(api_key=API_KEY)

MODEL = "llama-3.1-8b-instant"

print(f"Groq client configured with model: {MODEL}")

def ask_llm(user_message,
            system_message="You are a helpful assistant.",
            temperature=0.7,
            max_tokens=500):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content


test_response1 = ask_llm(
    "What is ETL in data engineering? Answer in exactly 2 sentences."
)

test_response2 = ask_llm(
    "Is GenAI and Data Engineering a good career in 2026?"
)

print("=== LLM Response ===")
print(test_response1)
print()
print(test_response2)

Groq client configured with model: llama-3.1-8b-instant
=== LLM Response ===
ETL (Extract, Transform, Load) is a fundamental process in data engineering that involves extracting data from various sources, transforming it into a standardized format, and loading it into a target system, such as a data warehouse or database. The ETL process is used to integrate data from multiple sources, ensure data quality and consistency, and facilitate data analysis and reporting.

As of my cut-off knowledge in 2023, GenAI (Generative AI) and Data Engineering are indeed promising and in-demand fields. However, I'll provide an analysis based on trends and potential growth prospects.

**GenAI:**

GenAI is an emerging field that combines AI, Machine Learning (ML), and data generation. With the rapid growth of AI, GenAI is expected to play a crucial role in various industries, including:

1. **Content creation**: GenAI can generate high-quality content, such as images, videos, music, and text, which can b

In [7]:
response_etl=ask_llm("In 3 bullet points,explain how the Medallion Architecture"
                    "(Bronze,zSilver,Gold: layers)relates to ETL pipelines",
                    system_message="You are a senior data engineering instructor" "Be concise and practical")
print("Medallion+ETL connection")
print(response_etl)
print()
print("----Token explanation---")
print("Each word is roughly 1-2 tokens:")
print("The model above used approxiamtely",len(response_etl.split()),"tokens")
print(" llama-3.1-8b context window;8192 tokens~ 6000 words per conversation")

Medallion+ETL connection
Here are three bullet points explaining the Medallion Architecture and its relation to ETL pipelines:

• **Data Ingestion (Bronze Layer)**: The Medallion Architecture's Bronze layer represents the initial data ingestion process, similar to the Extract phase in ETL. In this stage, raw data from various sources (e.g., databases, files, APIs) is collected and formatted for further processing. This aligns with the ETL pipeline's Extract phase, where data is pulled from various sources.

• **Data Processing and Enrichment (Silver Layer)**: The Medallion Architecture's Silver layer represents data processing and enrichment, which is akin to the Transform phase in ETL. In this stage, the raw data is cleaned, transformed, and enriched for analysis. This aligns with the ETL pipeline's Transform phase, where data is converted into a suitable format for analysis.

• **Data Warehousing and Read-Optimization (Gold Layer)**: The Medallion Architecture's Gold layer represents

In [8]:
zero_shot_response=ask_llm(
    "Extract the city name from this address: "
    "456 Brigade Road,Bangalore 560025,Karnataka,Idia"
)
print('zero-shot Result: ')
print(zero_shot_response)
print()

ambiguous_response=ask_llm("clean this data: ramesh Kumar,45000,mumbai")
print('Ambiguous zero-shot Result:')
print(ambiguous_response)
print()
print('problem:output format is unpredictable and not maachine parseable')

zero-shot Result: 
The city name from the given address is "Bangalore". However, I would like to point out a possible error in the address. It seems to be "India" instead of "India" being "Karnataka" is in India, but the word "Idia" is likely a typo and should be spelled "India".

Ambiguous zero-shot Result:
The given data seems to be a record containing a person's name, salary, and location. Here's a cleaned version of the data:

- **Name:** Ramesh Kumar
- **Salary:** 45,000
- **Location:** Mumbai

If this data is part of a larger dataset, I would suggest splitting the name into first and last names for easier querying and analysis:

- **First Name:** Ramesh
- **Last Name:** Kumar
- **Salary:** 45,000
- **Location:** Mumbai

However, without more context, it's difficult to determine the best way to structure this data.

problem:output format is unpredictable and not maachine parseable


In [9]:
few_shot_prompt="""
convert employee text to JSON.here are examples:
Input:RAMESH KUMAR,45000,mumbai
Output:{"name":"RAMESH KUMAR","salary":45000,"city":"mumbai"}

Input:Priya nair,52000,Delhi
Output:{"name":"Priya nair","salary":52000,"city":"Delhi"}
Now convert this:
Input:ANANYA DAS,38000,kolkata
Output:
"""
few_shot_response=ask_llm(few_shot_prompt)
print('few-shot Result:')
print(few_shot_response)
print()

try:
  parsed=json.loads(few_shot_response.strip())
  print("sucessfully parsed as JSON")
  print(f'Name: {parsed["name"]}')
  print(f'Salary: {parsed["salary"]}')
  print(f'City: {parsed["city"]}')
except json.JSONDecodeError:
  print('parsing failed --model added extra text')
  print('solution:add explicit instructions in the styles prompt')

few-shot Result:
To convert the employee text to JSON, you can use the following Python code:

```python
import json

def convert_to_json(employee_text):
    # Split the employee text into name, salary, and city
    employee_info = employee_text.replace(",", " ").split()
    name = " ".join(employee_info[:employee_info.index(' ')])
    salary = int(employee_info[employee_info.index(' ') + 1])
    city = " ".join(employee_info[employee_info.index(' ') + 2:])

    # Create a dictionary with the employee information
    employee_dict = {
        "name": name,
        "salary": salary,
        "city": city
    }

    # Convert the dictionary to JSON
    json_output = json.dumps(employee_dict)

    return json_output

# Test the function
employee_text = "ANANYA DAS,38000,kolkata"
print(convert_to_json(employee_text))
```

Output:
```json
{"name": "ANANYA DAS", "salary": 38000, "city": "kolkata"}
```

Alternatively, you can use a more robust solution that handles edge cases and multiple empl

In [10]:
same_question="review this python code and identify any issues:\n"\
"df['revenue]=df['qty']*df['price']\n"\
"result=df.groupby('dept).sum()"

generic_response=ask_llm(same_question,temperature=0.2)
print('without role prompting: ')
print(generic_response[:300],'...')
print()

role_response=ask_llm(
    same_question,
    system_message="you are a senior data engineer with 18 yrs of prediction"
    "experience.review code critically for production readiness,"
    "data type issues,and potential failures at scale.",
    temperature=0.2
)
print('with role promting(senior data engieer): ')
print(role_response[:400],'...')
print()
print('notice: role prompting produces more technical,actionablr feedback')

without role prompting: 
There are several issues with the provided Python code:

1. **Typo in column name**: The column name is misspelled as `df['revenue']` instead of `df['revenue']` (assuming it should be `df['revenue']`).

2. **Missing closing parenthesis**: The `groupby` function is missing a closing parenthesis. It s ...

with role promting(senior data engieer): 
Here's a review of the provided Python code:

```python
# Potential issue: missing colon at the end of the line
df['revenue'] = df['qty'] * df['price']

# Potential issue: missing closing parenthesis at the end of the line
result = df.groupby('dept').sum()
```

Here are some suggestions for improvement:

1. **Missing colon**: The line `df['revenue]=df['qty']*df['price']` is missing a colon at the  ...

notice: role prompting produces more technical,actionablr feedback


In [11]:
prompt="give me one creative name for a data analytics startup."

print('===Temperature Experiment===')
for temp in [0.0,0.5,1.0]:
  response=ask_llm(prompt,temperature=temp)
  print(f'temp={temp}: {response.strip()}')
  time.sleep(1)
print()
print('observation: ')
print( 'temperature=0.0->same or very similar answer every run (deterministic)')
print( 'temperature=0.5->some variation')
print( 'temperature=1.0 ->more creative/varied,sometimes surpricing')
print()
print('rule for data engineering tasks:use temperature=0.0 or 0.1')
print('you need CONSISTENT,PARSEABLE output -> not creative variation')

===Temperature Experiment===
temp=0.0: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" suggests connection and linkages, implying the ability to connect disparate data points and provide valuable insights. This name conveys the idea of a startup that helps businesses navigate complex data landscapes and uncover hidden patterns and trends.
temp=0.5: Here's a creative name for a data analytics startup:

**Nexa Insights**

"Nexa" comes from the word "nexus," meaning a connection or link between things. This name suggests that the startup helps connect data points to provide valuable insights, making it a fitting name for a data analytics company.
temp=1.0: "Purspectum" - a combination of "per-spect" and "spectrum," suggesting a company that offers perspective on data through its analytical tools and services, and a wide range of offerings across various industries and sectors.

observation: 
temperature=0.0->same or very similar answer every run (determinis

In [14]:
invoice_test="Invoice #2024-001 from TECHWORLD SOLUTIONS dated 15th January 2024.Amount:Rs.45,000 for Laptop"
weak_response=ask_llm(
    f'clean this invoice data: {invoice_test}',
    temperature=0.3
)
print('weak prompt output: ')
print(weak_response)
print()

try:
  json.loads(weak_response)
  print('PARSABLE: yes')

except:
  print('PARSABLE: no-canot load into dataframe')
print('\n'+'='*50+'\n')

strong_system="""You are a data extraction specialist for an accounting pipeline.
extract invoice data and return ONLY a valid JSON object.
do not include any explanations,preamble,or markdown formatting.
return only the json,nothing else.

JSON schema (use null for missing values):
{"invoice_id":string,"vendor_name":string(Title Case),"amount":number(no currency symbols),
"currency":string(default INR),
"invoice_date":string(YYYY-MM-DD),
"category":string(Electronics/Services/Accessories/other)}"""

strong_response=ask_llm(
    f"Extract from :{invoice_test}",
    system_message=strong_system,
    temperature=0.0 #Always 0 for structured data extraction
)
print('STONG PROMPT OUTPUT:')
print(strong_response)
print()
try:
  parsed=json.loads(strong_response.strip())
  print('PARSEABLE:Yes')
  print(f"Vendor:{parsed.get('vendor_name')}")
  print(f'Amount:{parsed.get('amount')}')
  print(f'Date :{parsed.get("invoice_date")}')
except json.JSONDecodeError:
  #Fallback extract JSON with re.Regex
  match=re.search(r'\{.*?\}',strong_response,re.Dotall)
  if match:
    parsed=json.loads(match.group())
    print('PARSEABLE:Yes (extracted with regex fallback)')
  else:
    print('PAREABLE :No -retry with stricter prompt')





weak prompt output: 
Here's the cleaned invoice data:

**Invoice Information:**

- **Invoice Number:** 2024-001
- **Date:** 15th January 2024
- **Company:** TECHWORLD SOLUTIONS
- **Amount:** Rs. 45,000
- **Item Purchased:** Laptop

This data is now organized and easily readable.

PARSABLE: no-canot load into dataframe


STONG PROMPT OUTPUT:
{"invoice_id":"2024-001","vendor_name":"Techworld Solutions","amount":45000,"currency":"INR","invoice_date":"2024-01-15","category":"Electronics"}

PARSEABLE:Yes
Vendor:Techworld Solutions
Amount:45000
Date :2024-01-15


#MINI PROJECT

GOAL:Convert 5 messy invoice strings into a clean,structured pandas dataframe using LLM.
this is a complete GenAI-powered ETL pipeline:

messy text->LLM->JSON->DataFrame->Analysis

In [15]:
#install required libraries and import libraries
#!pip install groq pandas
#import pandas as pd
#from groq import groq

In [16]:
#add api key
API_KEY = "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"

client = Groq(api_key=API_KEY)

In [17]:
invoices = [

    "Invoice#101 John bought 2 laptops for $800 each on 12/01/2025",

    "Bill No 202 | Customer: Alice | Product: Mobile | Qty=1 | Total=600 dollars",

    "INV-303 Mike ordered 5 headphones worth $50 each",

    "Receipt 404: Sarah purchased 3 keyboards amount 120 each",

    "Invoice 505 customer David bought 2 monitors total price 400"
]

In [18]:
def extract_invoice(invoice_text):

    prompt = f"""
    Extract invoice information from the text below.

    Return ONLY valid JSON.
    Do not add explanation.
    Do not use markdown.
    Do not write anything except JSON.

    Required fields:
    - invoice_id
    - customer_name
    - product
    - quantity
    - unit_price
    - total_price

    Invoice:
    {invoice_text}
    """

    response = client.chat.completions.create(

        model="llama-3.1-8b-instant",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [19]:
#test on one invoice
result=extract_invoice(invoices[0])
print(result)

{"invoice_id": "101", "customer_name": "John", "product": "laptops", "quantity": "2", "unit_price": "800", "total_price": "1600"}


In [20]:
structured_data = []

for invoice in invoices:

    result = extract_invoice(invoice)

    print("\nRAW OUTPUT:")
    print(result)

    try:

        # clean markdown if exists
        cleaned_result = result.replace("```json", "") \
                               .replace("```", "") \
                               .strip()

        json_data = json.loads(cleaned_result)

        structured_data.append(json_data)

    except Exception as e:

        print("Error parsing:", invoice)
        print("Reason:", e)


RAW OUTPUT:
{
  "invoice_id": 101,
  "customer_name": "John",
  "product": "laptop",
  "quantity": 2,
  "unit_price": 800,
  "total_price": 1600
}

RAW OUTPUT:
{"invoice_id": "202", "customer_name": "Alice", "product": "Mobile", "quantity": 1, "unit_price": 600, "total_price": 600}

RAW OUTPUT:
    {
    "invoice_id": "INV-303",
    "customer_name": "Mike",
    "product": "headphones",
    "quantity": 5,
    "unit_price": 50,
    "total_price": 250
    }

RAW OUTPUT:
{
  "invoice_id": "Receipt 404",
  "customer_name": "Sarah",
  "product": "keyboards",
  "quantity": 3,
  "unit_price": 120,
  "total_price": 360
}

RAW OUTPUT:
{
  "invoice_id": 505,
  "customer_name": "David",
  "product": "monitors",
  "quantity": 2,
  "unit_price": 200,
  "total_price": 400
}


In [21]:
#check extracted data
structured_data

[{'invoice_id': 101,
  'customer_name': 'John',
  'product': 'laptop',
  'quantity': 2,
  'unit_price': 800,
  'total_price': 1600},
 {'invoice_id': '202',
  'customer_name': 'Alice',
  'product': 'Mobile',
  'quantity': 1,
  'unit_price': 600,
  'total_price': 600},
 {'invoice_id': 'INV-303',
  'customer_name': 'Mike',
  'product': 'headphones',
  'quantity': 5,
  'unit_price': 50,
  'total_price': 250},
 {'invoice_id': 'Receipt 404',
  'customer_name': 'Sarah',
  'product': 'keyboards',
  'quantity': 3,
  'unit_price': 120,
  'total_price': 360},
 {'invoice_id': 505,
  'customer_name': 'David',
  'product': 'monitors',
  'quantity': 2,
  'unit_price': 200,
  'total_price': 400}]

In [22]:
#create dataframe
df=pd.DataFrame(structured_data)
df

,invoice_id,customer_name,product,quantity,unit_price,total_price
0,101,John,laptop,2,800,1600
1,202,Alice,Mobile,1,600,600
2,INV-303,Mike,headphones,5,50,250
3,Receipt 404,Sarah,keyboards,3,120,360
4,505,David,monitors,2,200,400


In [23]:
#Data Cleaning
df["quantity"] = pd.to_numeric(df["quantity"])

df["unit_price"] = pd.to_numeric(df["unit_price"])

df["total_price"] = pd.to_numeric(df["total_price"])

In [24]:
#analysis
print("Total Revenue: ",df["total_price"].sum())

Total Revenue:  3210


In [25]:
#highest invoice
df.sort_values(by="total_price", ascending=False)

,invoice_id,customer_name,product,quantity,unit_price,total_price
0,101,John,laptop,2,800,1600
1,202,Alice,Mobile,1,600,600
4,505,David,monitors,2,200,400
3,Receipt 404,Sarah,keyboards,3,120,360
2,INV-303,Mike,headphones,5,50,250


In [26]:
#average invoice value
print("Average Invoice Value:", df["total_price"].mean())

Average Invoice Value: 642.0


In [27]:
#save csv
df.to_csv("clean_invoices.csv",index=False)

MINI PROJECT: Smart Data Cleaner

Goal: Convert 5 messy invoice strings into a clean, structured Pandas Dataframe using LLM

This is a complete GenAI-powered ETL pipeline.

Messy Text-> LLM -> JSON -> DataFrame ->Analysis

Q1. What is the difference between ML (day 5) and Generative AI(Day 6)

Q2. What does temperature=0.0 do in an LLM API call and when would you use it?

Q3. Write a few-shot prompt that extracts name and salary from text in JSON format.

Q4. What is LLM hallucination and how can prompt engineering reduce it?

Q5. Your LLM returns ```json\n{"name":"Ramesh"}\n and json.loads() crashes. Write the fix.

*Q6:*How does today's


Q1. Difference between ML and Generative AI
ML focuses on learning patterns from data to make predictions or decisions. Generative AI is a type of ML that creates new, original content (like text or images) based on learned patterns.

Q2. What does temperature=0.0 do?
temperature=0.0 makes the LLM's output highly deterministic and consistent (least creative). Use it for tasks requiring precise, repeatable results, such as data extraction and parsing.

Q3. Few-shot prompt for name and salary extraction in JSON

In [28]:
few_shot_prompt_short = '''
Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:
'''
print('Short Few-Shot Prompt:')
print(few_shot_prompt_short)
# Expected output for 'ALICE JOHNSON, 82000' would be: {"name": "Alice Johnson", "salary": 82000}

Short Few-Shot Prompt:

Convert employee text to JSON. Examples:
Input: John Doe, 60000
Output: {"name": "John Doe", "salary": 60000}

Input: Jane Smith, 75000
Output: {"name": "Jane Smith", "salary": 75000}

Input: ALICE JOHNSON, 82000
Output:



Q4. LLM hallucination and how prompt engineering reduces it
LLM hallucination is when the model generates false or nonsensical information. Prompt engineering reduces it by providing clear instructions, context, examples, and strict output constraints (e.g., "Answer only from provided text", "Return ONLY JSON").

Q5. Fix for json.loads() crashing on ```json\n{"name":"Ramesh"}\n

In [29]:
import json
import re

llm_output_bad = '```json\n{"name":"Ramesh"}\n```'

# Use regex to extract the pure JSON string
match = re.search(r'```json\n(.*)\n```', llm_output_bad, re.DOTALL)
if match:
    json_string_fixed = match.group(1).strip()
    try:
        parsed_data = json.loads(json_string_fixed)
        print(f"Fixed: Successfully parsed as: {parsed_data}")
    except json.JSONDecodeError as e:
        print(f"Error even after regex: {e}")
else:
    print("No JSON block found.")

Fixed: Successfully parsed as: {'name': 'Ramesh'}


Q6. Smart Cleaner vs. Manual ETL Cleaning
Today's Smart Cleaner (LLM-based) automatically extracts and structures data from unstructured text using natural language understanding, adapting flexibly to variations. Manual ETL relies on predefined rules for structured/semi-structured data and requires more effort to adapt to new formats.

In [30]:
# Make sure to run all preceding cells, especially the one defining `ask_llm`.
few_shot_prompt = """
Convert employee text to JSON. Here are examples:

Input: RAMESH KUMAR, 45000, mumbai
Output: {"name": "Ramesh Kumar", "salary": 45000, "city": "Mumbai"}

Input: priya nair, 52000, Delhi
Output: {"name": "Priya Nair", "salary": 52000, "city": "Delhi"}

Now convert this:
Input: ANANYA DAS, 38000, kolkata
Output:
"""
few_shot_response = ask_llm(few_shot_prompt, temperature=0.0)
print('Few-Shot Result:')
print(few_shot_response)
print()

try:
  # The type hint 'parsed: json.loads' is incorrect. It should be 'parsed = json.loads'
  parsed = json.loads(few_shot_response.strip())
  print('Sucessfully parsed as JSON!')
  print(f'Name: {parsed["name"]}, Salary: {parsed["salary"]}, City: {parsed["city"]}')

except json.JSONDecodeError:
  print('Parsing failed - model added extra text or invalid JSON.')
  print("Solution: add explicit instructions in the system prompt to return ONLY JSON, and check your prompt examples for correct JSON syntax.")


Few-Shot Result:
Here's a Python function that can convert the employee text to JSON:

```python
import json

def convert_to_json(employee_text):
    # Split the input string into individual values
    values = employee_text.split(', ')
    
    # Create a dictionary with the given keys
    employee_data = {
        "name": values[0].strip().title(),
        "salary": int(values[1]),
        "city": values[2].strip().title()
    }
    
    # Convert the dictionary to JSON
    json_data = json.dumps(employee_data)
    
    return json_data

# Test the function
employee_text = "ANANYA DAS, 38000, kolkata"
print(convert_to_json(employee_text))
```

When you run this function with the input "ANANYA DAS, 38000, kolkata", it will output:

```json
{"name": "Ananya Das", "salary": 38000, "city": "Kolkata"}
```

This function works by splitting the input string into individual values using the `split(', ')` method. It then creates a dictionary with the given keys and assigns the corresponding v